# 第 4 章:注意力原子 —— RMSNorm + RoPE + 单头注意力

在前三章里,我们把 minimind 跑了起来(第 1 章),学会了把文本切成 token(第 2 章),弄清了每个超参数为什么这么选(第 3 章)。

现在,我们终于要打开 `model_minimind.py` 的**核心引擎** —— 注意力机制。

本章的写法是 **naive → compact**:先用最直白、最笨的代码把每个概念写一遍,让你彻底理解「它在算什么」,然后把它们一步步压扁、合并,最终变成 minimind 源码里那 44 行紧凑的 `Attention` 类(`model_minimind.py:~96-139 (@67f114a)`)。

> 「注意力」不是一个东西,而是**一整套零件**:RMSNorm、RoPE、causal mask、multi-head、GQA、QK-Norm、KV cache。本章逐个拆。

## 4.0 本章路线图

minimind 的注意力层(44 行)做了 **7 件事**,每件事都对应本章一节:

| 节 | 概念 | 作用 | 对应源码行 |
|---|---|---|---|
| 4.1 | RMSNorm | 归一化(注意力的前置条件) | `50-60` |
| 4.3 | naive self-attention | 「词与词之间关注多少」的最原始实现 | — |
| 4.4 | trainable Wq/Wk/Wv | 用线性层把输入投影成 Q/K/V | `100-103` |
| 4.5 | causal mask | 让每个位置只能看到过去 | `129` |
| 4.6 | multi-head | 把 d 维拆成多个头并行算注意力 | `114-116` |
| 4.7 | RoPE | 用旋转注入位置信息 | `62-84` |
| 4.8 | GQA | Q 头比 KV 头多,省显存 | `86-89, 94-97` |
| 4.9 | QK-Norm | 对 Q/K 的 head_dim 再做 RMSNorm | `104-105, 117` |
| 4.10 | KV cache | 生成时缓存历史 K/V | `120-123` |
| 4.11 | compact Attention | 全部合并成最终实现 | `91-134` |

> 代码风格要求:本章 **每一个 shape 变化都有注释**。形状是理解注意力的钥匙。

&nbsp;

---

## Part 1:RMSNorm —— 归一化是注意力的前提

### 4.1 为什么需要 Normalization

在算注意力之前,输入向量需要先「归一化」。原因:如果某些维度的数值特别大,经过 `Q @ K^T` 后点积会爆炸,softmax 输出变成 one-hot(某个词独占全部注意力),梯度消失。

minimind 用的是 **RMSNorm**(Root Mean Square Normalization),比 LayerNorm 简单:不需要减均值,只除以 RMS(均方根)。

公式:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot \gamma$$

其中 $\gamma$ 是可学习的缩放向量(`self.weight`),$\epsilon$ 防止除零。

### 4.2 RMSNorm 的 naive → compact 写法

先看最直白的展开写法,再看 minimind 的单行实现:

In [ ]:
import torch

# === naive RMSNorm:展开每一步 ===
def rmsnorm_naive(x, weight, eps=1e-6):
    # x: (b, T, d) — batch, seq_len, dim
    d = x.shape[-1]
    # Step 1: 算每个位置所有维度的平方均值
    sq_mean = (x ** 2).mean(dim=-1, keepdim=True)       # Shape: (b, T, d) -> (b, T, 1)
    # Step 2: 加 eps 再开根号的倒数
    inv_rms = 1.0 / torch.sqrt(sq_mean + eps)            # Shape: (b, T, 1) -> (b, T, 1)
    # Step 3: 归一化
    x_normed = x * inv_rms                               # Shape: (b, T, 1) * (b, T, d) -> (b, T, d)
    # Step 4: 乘以可学习权重 gamma
    return x_normed * weight                             # Shape: (b, T, d) * (d,) -> (b, T, d)

# 测试:3 个 token,每个 4 维
x = torch.tensor([
    [[1.0, 2.0, 3.0, 4.0]],   # token 0
    [[5.0, 6.0, 7.0, 8.0]],   # token 1
])  # Shape: (2, 1, 4)
weight = torch.ones(4)
print("naive output:", rmsnorm_naive(x, weight))
# naive output: tensor([[[0.3651, 0.7303, 1.0954, 1.4606]],
#                       [[2.2555, 2.7066, 3.1577, 3.6088]]])

In [ ]:
# === compact RMSNorm:就是 minimind 的写法(model_minimind.py:~61-65 (@67f114a)) ===

def rmsnorm_compact(x, weight, eps=1e-6):
    # 整个 norm 函数压缩成一行:
    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)  # Shape: (b,T,d) -> (b,T,d)
    return weight * normed.float().type_as(x)                        # upcast 后再缩回原精度

print("compact output:", rmsnorm_compact(x, weight))
print("差异:", (rmsnorm_naive(x, weight) - rmsnorm_compact(x, weight)).abs().max().item())
# 差异: 0.0 (完全相同)

> **`torch.rsqrt`** = `1 / torch.sqrt` 的优化版,省一次除法。minimind 还用 `.float()` 先上转到 fp32 计算(数值稳定),再 `.type_as(x)` 转回原精度(half/bf16),避免 fp16 溢出。

注意 RMSNorm **不减均值**(`- mean`),比 LayerNorm 少了约 7% 的计算量(因为 mean 和减法都需要对整个维度做一次 reduce)。对于 LLM 级别的计算量,这个省法很值得。

---

&nbsp;

## Part 2:Self-Attention 的 naive → compact(本章核心)

这是整章最重要的部分。我们从零开始,用最笨的代码实现注意力,然后一步步逼近 minimind 的最终版本。

### 4.3 Naive Self-Attention:不加任何权重

「注意力」的本质很简单:给定一序列的向量,**让每个向量去看其他所有向量,然后加权平均**。

最朴素的版本 —— 每个向量**既是 Q 又是 K 又是 V**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

分三步:
1. **打分**:Q 和 K 做点积,得到「每个词对每个词的关注程度」
2. **归一化**:softmax 把分数变成概率(加起来 = 1)
3. **加权求和**:用概率对 V 加权平均,得到输出

In [ ]:
import torch.nn.functional as F

torch.manual_seed(42)
d = 4  # 每个向量 4 维

# 3 个 token 的输入向量(假装是 embedding)
# Shape: (T, d) = (3, 4)
x = torch.randn(3, 4)

# === 最 naive 的 self-attention:Q = K = V = x ===
Q = x  # Shape: (T, d) = (3, 4)
K = x  # Shape: (T, d) = (3, 4)
V = x  # Shape: (T, d) = (3, 4)

# Step 1: Q @ K^T 打分 —— 得到 T×T 的注意力矩阵
scores = Q @ K.transpose(-2, -1)   # Shape: (T,d) @ (d,T) -> (T,T) = (3,3)
print("raw scores:\n", scores)

# Step 2: 缩放 —— 除以 sqrt(d_k) 控制数值范围
scores = scores / (d ** 0.5)       # Shape: (T,T) -> (T,T)
print("scaled scores:\n", scores)

# Step 3: softmax 归一化 —— 每行加起来 = 1
attn_weights = F.softmax(scores, dim=-1)  # Shape: (T,T) -> (T,T)
print("attn weights:\n", attn_weights)
print("每行之和:", attn_weights.sum(dim=-1))  # 应该都是 1.0

# Step 4: 加权求和 —— 用注意力权重对 V 加权
output = attn_weights @ V           # Shape: (T,T) @ (T,d) -> (T,d) = (3,4)
print("output:\n", output)
print("output shape:", output.shape)  # (3, 4) — 和输入一样!

注意输出的 shape 和输入完全一样:`(3, 4) → (3, 4)`。这是注意力的重要特性 —— **它不改变维度,只改变内容**。

每个 token 的输出是所有 token 的加权和,权重由相似度决定。token 和自己最相似,所以对角线上的权重最大。

> **为什么要除以 $\sqrt{d_k}$?** 因为点积的方差随维度增长。维度越大,scores 的数值越大,softmax 就越接近 one-hot。除以 $\sqrt{d_k}$ 把方差稳定在 1 附近,让 softmax 有合理的「平滑度」。

### 理解注意力矩阵的直觉

把 `attn_weights` 想成一张 $T \times T$ 的热力图 —— **第 $i$ 行第 $j$ 列**表示「token $i$ 对 token $j$ 的关注程度」。对角线(自己看自己)通常最亮,因为一个向量和自己的点积最大。

```
          token_0  token_1  token_2
token_0  [ 0.40    0.20    0.40 ]   ← token 0 的注意力分布
token_1  [ 0.20    0.40    0.40 ]
token_2  [ 0.30    0.30    0.40 ]   ← 对角线偏亮
```

每一行加起来 = 1(softmax 保证)。输出 $O_i = \sum_j \text{attn}_{ij} \cdot V_j$ 就是「token $i$ 根据关注度从所有 token 的 V 中取信息」。

> **一句话**:注意力 = 「每个词从所有词的 Value 里按相似度比例取一点」。

### 4.4 加入可训练权重:Wq / Wk / Wv / Wo

naive 版本的问题是:**所有层、所有 token 都用完全相同的 Q=K=V**,模型没有任何可学习的自由度。真实的注意力需要**把输入投影成不同的 Q/K/V**。

这就是 `Wq`、`Wk`、`Wv` 三个线性层的作用:

In [ ]:
import torch.nn as nn

d_model = 4  # 输入维度
d_k = 4      # Q/K/V 维度(这里简化成和 d_model 一样)

# 三个可学习的线性投影(bias=False,LLM 惯例)
Wq = nn.Linear(d_model, d_k, bias=False)  # Shape: (d_model, d_k)
Wk = nn.Linear(d_model, d_k, bias=False)  # Shape: (d_model, d_k)
Wv = nn.Linear(d_model, d_k, bias=False)  # Shape: (d_model, d_k)
Wo = nn.Linear(d_k, d_model, bias=False)  # Shape: (d_k, d_model) — 输出投影

x = torch.randn(3, d_model)  # Shape: (T, d_model) = (3, 4)

# 投影成 Q/K/V —— 现在每个 token 有三个不同的角色
Q = Wq(x)  # Shape: (T, d_model) -> (T, d_k) = (3, 4)
K = Wk(x)  # Shape: (T, d_model) -> (T, d_k) = (3, 4)
V = Wv(x)  # Shape: (T, d_model) -> (T, d_k) = (3, 4)

scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)  # Shape: (T,d_k)@(d_k,T) -> (T,T)
attn = F.softmax(scores, dim=-1)                   # Shape: (T,T) -> (T,T)
output = Wo(attn @ V)                              # Shape: (T,T)@(T,d_k) -> (T,d_k) -> Wo -> (T, d_model)
print("with Wq/Wk/Wv/Wo, output shape:", output.shape)  # (3, 4)

# 现在有 4 个矩阵要训练!模型可以学习「该关注谁」
print(f"可训练参数量: {sum(p.numel() for p in [Wq, Wk, Wv, Wo])}")  # 4 * 4 * 4 = 64

### 4.5 Causal Masking:让注意力「不能偷看未来」

上面的版本有一个致命问题:**第 1 个 token 能看到第 3 个 token**。但 LLM 是自回归的 —— 生成第 $t$ 个 token 时,你**还没有**第 $t+1$ 个 token。

解决方案:**因果掩码(causal mask)**。用上三角矩阵把「未来」的位置遮成 $-\infty$,softmax 后变成 0:

In [ ]:
T = 3  # 序列长度

# 构造下三角掩码:位置 i 只能看位置 0..i
mask = torch.tril(torch.ones(T, T))   # Shape: (T, T) = (3, 3)
print("causal mask (1=可见, 0=遮蔽):")
print(mask)
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])

# 用 -inf 替换 0,这样 softmax 后这些位置就是 0
mask_inf = mask.masked_fill(mask == 0, float('-inf'))  # Shape: (T,T) -> (T,T)
print("\n-inf mask:")
print(mask_inf)
# tensor([[0., -inf, -inf],
#         [0., 0., -inf],
#         [0., 0., 0.]])

# 加到 scores 上
scores_masked = scores + mask_inf  # Shape: (T,T) + (T,T) -> (T,T)
attn_masked = F.softmax(scores_masked, dim=-1)  # Shape: (T,T) -> (T,T)
print("\nmasked attention weights:")
print(attn_masked)
# 第 0 行只有第 0 列非零,第 1 行有前 2 列,第 2 行有全部 3 列

> minimind 的实现更精简(`model_minimind.py:~134 (@67f114a)`):
> ```python
> scores[:, :, :, -seq_len:] += torch.full((T, T), float("-inf")).triu(1)
> ```
> 用 `triu(1)` 直接生成上三角(对角线以上),不需要先建全 1 矩阵再替换。`triu(1)` 的 `1` 表示从第 1 条对角线开始(不含主对角线)。

现在我们有了完整的 **「单头因果自注意力」**。但真实 LLM 不止一个头 —— 接下来把它变成多头。

### Part 2 小结:naive → 单头因果注意力的三步演化

| 版本 | 加了什么 | 公式变化 |
|---|---|---|
| 4.3 naive | 无 | $O = \text{softmax}(xx^T/\sqrt{d})x$ |
| 4.4 +权重 | Wq/Wk/Wv/Wo | $O = W_o \cdot \text{softmax}(W_q x \cdot (W_k x)^T / \sqrt{d}) \cdot W_v x$ |
| 4.5 +mask | causal | scores 上加上三角 $-\infty$ |

每一步只加了一层抽象,核心计算还是 $QK^T \to \text{softmax} \to V$ 这条流水线。multi-head 也不会改变这个核心 —— 它只是把 $d$ 维拆成多份并行算。

&nbsp;

---

## Part 3:Multi-Head + RoPE + GQA + QK-Norm

### 4.6 Multi-Head Attention:把维度拆成多个头

「单头注意力」只能学到一种关注模式。但语言里有多种关系:语法依赖、指代消解、语义相似…… 一个头不够。

**Multi-Head** 的做法:把 $d$ 维向量拆成 $n_h$ 个头,每个头维度 $d_h = d / n_h$,各自独立算注意力,最后拼回来。

minimind 的参数:`hidden_size=768`, `num_attention_heads=8`, `head_dim=96`(注意 $768 / 8 = 96$)。

In [ ]:
# 模拟 multi-head 的 shape 变化(minimind: d=768, n_h=8, d_h=96)
b, T = 2, 5        # batch=2, seq_len=5
d_model = 768
n_heads = 8
d_h = d_model // n_heads  # 96

# 输入:(batch, seq_len, d_model)
x = torch.randn(b, T, d_model)

# 一次投影出所有头的 Q(而不是 8 个独立的 Linear)
Wq = nn.Linear(d_model, n_heads * d_h, bias=False)  # Shape: (768, 768)
Q = Wq(x)  # Shape: (b, T, d_model) -> (b, T, n_heads * d_h) = (2, 5, 768)

# 关键:reshape 把最后一维拆成 (n_heads, d_h)
Q = Q.view(b, T, n_heads, d_h)  # Shape: (b,T,768) -> (b, T, n_heads, d_h) = (2, 5, 8, 96)

# transpose 把 head 维提到前面,方便批量矩阵乘
Q = Q.transpose(1, 2)  # Shape: (b, T, n_h, d_h) -> (b, n_h, T, d_h) = (2, 8, 5, 96)

# K/V 做同样的事
K = Wk(x).view(b, T, n_heads, d_h).transpose(1, 2)  # (b,T,d) -> (b,n_h,T,d_h)
V = Wv(x).view(b, T, n_heads, d_h).transpose(1, 2)  # (b,T,d) -> (b,n_h,T,d_h)

# 批量注意力:所有头同时算
scores = Q @ K.transpose(-2, -1) / (d_h ** 0.5)  # Shape: (b,n_h,T,d_h)@(b,n_h,d_h,T) -> (b,n_h,T,T)
attn = F.softmax(scores, dim=-1)                   # Shape: (b, n_h, T, T)
out = attn @ V                                     # Shape: (b,n_h,T,T)@(b,n_h,T,d_h) -> (b,n_h,T,d_h)

# 拼回:transpose + reshape
out = out.transpose(1, 2)           # Shape: (b,n_h,T,d_h) -> (b, T, n_h, d_h)
out = out.reshape(b, T, -1)         # Shape: (b, T, n_h, d_h) -> (b, T, n_h*d_h) = (b, T, 768)
print("multi-head output shape:", out.shape)  # (2, 5, 768) — 回到 d_model!

> **shape 是这里的全部。** 记住这个流程:
> ```
> (b, T, d) → view → (b, T, n_h, d_h) → transpose → (b, n_h, T, d_h)
> ```
> 然后 `Q @ K^T` 在最后两维上做,前面 `b` 和 `n_h` 维度被当作 batch 广播。这就是 PyTorch 批量矩阵乘法 `@` 的魅力 —— 8 个头的注意力**一步算完**。

### 4.7 RoPE:旋转位置编码

到目前为止,我们的注意力有一个问题:**它不知道 token 的顺序**。交换两个 token 的位置,Q/K 的点积不变(因为 `Q @ K^T` 对行排列不敏感)。

**RoPE(Rotary Position Embedding)** 的做法:对 Q 和 K 的每一对维度做旋转,旋转角度随位置增大。这样**相对位置**就被编码进了点积中。

公式(对每一对维度 $(x_{2i}, x_{2i+1})$):

$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

其中 $m$ 是位置,$\theta_i = \text{rope\_base}^{-2i/d}$,minimind 的 `rope_theta = 1e6`。

In [ ]:
# RoPE 的三个步骤(对应 model_minimind.py:~67-89 (@67f114a))

d_h = 96       # head_dim
max_len = 32768  # max_position_embeddings
rope_theta = 1e6

# Step 1: 预计算频率(precompute_freqs_cis, ~67-83 (@67f114a))
# theta_i = rope_theta^(-2i/d), i = 0,1,...,d/2-1
freqs = 1.0 / (rope_theta ** (torch.arange(0, d_h, 2)[: (d_h // 2)].float() / d_h))  # Shape: (d_h//2,) = (48,)
t = torch.arange(max_len)  # Shape: (max_len,) — 位置 0,1,2,...
freqs = torch.outer(t, freqs).float()  # Shape: (max_len, d_h//2) — 每个位置每对维度的角度

# cos/sin 并在最后一维复制一份(因为 rotate_half 要用)
cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)  # Shape: (max_len, d_h//2) -> (max_len, d_h)
sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)  # Shape: (max_len, d_h//2) -> (max_len, d_h)

print(f"cos shape: {cos.shape}")  # (32768, 96)
print(f"位置 0 的 cos[:4]: {cos[0, :4]}")  # 全是 1.0(角度=0,不旋转)
print(f"位置 1 的 cos[:4]: {cos[1, :4]}")  # 开始旋转

In [ ]:
# Step 2: rotate_half(apply_rotary_pos_emb 内部, ~86 (@67f114a))
# 把后半部分取负、和前半部分拼起来

def rotate_half(x):
    # x: (..., d_h)
    half = x.shape[-1] // 2
    return torch.cat((-x[..., half:], x[..., :half]), dim=-1)  # Shape: (...,d_h) -> (...,d_h)

# 例子:[a, b, c, d] -> [-c, -d, a, b]
demo = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
print("rotate_half([1,2,3,4]):", rotate_half(demo))  # [[-3, -4, 1, 2]]

# Step 3: 应用旋转(apply_rotary_pos_emb, ~85-89 (@67f114a))
def apply_rope(q, cos, sin):
    # q: (b, n_h, T, d_h), cos/sin: (T, d_h)
    cos = cos.unsqueeze(1)  # Shape: (T, d_h) -> (T, 1, d_h) — 广播到 head 维
    sin = sin.unsqueeze(1)  # Shape: (T, d_h) -> (T, 1, d_h)
    # 核心公式:q' = q * cos + rotate_half(q) * sin
    return (q * cos) + (rotate_half(q) * sin)  # Shape: (b,n_h,T,d_h) -> (b,n_h,T,d_h)

# 测试:位置 0 不旋转(因为 cos=1, sin=0)
q = torch.randn(1, 8, 5, 96)        # Shape: (b, n_h, T, d_h)
cos_pos = cos[:5]  # 取前 5 个位置                     # Shape: (5, 96)
sin_pos = sin[:5]  # Shape: (5, 96)
q_rotated = apply_rope(q, cos_pos, sin_pos)  # Shape: (b,n_h,T,d_h) -> (b,n_h,T,d_h)

print(f"位置 0 旋转前后差异: {(q[0,:,0,:] - q_rotated[0,:,0,:]).abs().max().item():.2e}")
# 应该接近 0(cos=1, sin=0,不旋转)
print(f"位置 1 旋转前后差异: {(q[0,:,1,:] - q_rotated[0,:,1,:]).abs().max().item():.4f}")
# 应该有明显变化

> **为什么 RoPE 用旋转?** 数学上可以证明:位置 $m$ 的 $Q$ 和位置 $n$ 的 $K$ 做点积,结果只依赖**相对距离** $m - n$。这意味着模型学到的注意力模式能泛化到训练时没见过的更长序列。这是 minimind 能把上下文扩展到 32768 的数学基础。
>
> **`rope_theta=1e6`** 比标准值 `10000` 大 100 倍。这意味着频率更低、旋转更慢,适合更长的上下文 —— minimind 用它配合 YaRN 外推(factor=16)把 2048 训练长度扩展到 32768。

### 4.8 GQA:Grouped-Query Attention

到此为止,我们假设 Q 和 KV 的头数一样多。但推理时 **KV cache 占显存**,KV 头越多越费。

**GQA** 的思路:Q 头保持 $n_h=8$ 个,但 KV 只用 $n_{kv}=4$ 个,每个 KV 头被 2 个 Q 头共享。省了一半 KV cache。

minimind:`num_attention_heads=8`,`num_key_value_heads=4`,`n_rep = 8 // 4 = 2`。

In [ ]:
# GQA 的核心:repeat_kv(~91-94 (@67f114a))
# 把 4 个 KV 头复制成 8 个,每个重复 n_rep=2 次

def repeat_kv(x, n_rep):
    b, T, n_kv, d_h = x.shape
    if n_rep == 1:
        return x
    # 插入一个维度,expand 复制,再 reshape
    x = x[:, :, :, None, :]  # Shape: (b, T, n_kv, d_h) -> (b, T, n_kv, 1, d_h)
    x = x.expand(b, T, n_kv, n_rep, d_h)  # Shape: (b,T,n_kv,1,d_h) -> (b,T,n_kv,n_rep,d_h)
    x = x.reshape(b, T, n_kv * n_rep, d_h)  # Shape: (b,T,n_kv,n_rep,d_h) -> (b,T,n_kv*n_rep,d_h)
    return x

# 测试:4 个 KV 头 → 8 个
kv = torch.randn(2, 5, 4, 96)   # Shape: (b, T, n_kv=4, d_h=96)
kv_expanded = repeat_kv(kv, n_rep=2)  # Shape: (b,T,4,d_h) -> (b,T,8,d_h)
print(f"KV before: {kv.shape}")        # (2, 5, 4, 96)
print(f"KV after repeat: {kv_expanded.shape}")  # (2, 5, 8, 96)

# 验证:第 0 个和第 1 个应该是同一个头(都被复制了)
print(f"head 0 == head 1? {(kv_expanded[:,:,0,:] == kv_expanded[:,:,1,:]).all().item()}")  # True
print(f"head 2 == head 3? {(kv_expanded[:,:,2,:] == kv_expanded[:,:,3,:]).all().item()}")  # True

> **GQA 省了多少?** KV cache 的显存 = `2 × n_kv × T × d_h × bytes`。从 `n_kv=8`(MHA)降到 `n_kv=4`(GQA),省一半。如果降到 `n_kv=1`(MQA),省 8 倍,但效果下降太多。GQA(2:1)是效果和效率的甜蜜点。
>
> **为什么用 `expand` 而不是 `repeat`?** `expand` 不复制数据,只创建视图(view),零内存开销。`reshape` 只是改 stride。两者合起来实现了「逻辑复制 + 物理零拷贝」。

### 4.9 QK-Norm:对 head_dim 再做一次 RMSNorm

这是 minimind 的一个细节:在投影出 Q/K 之后、应用 RoPE 之前,**对 head_dim 再做一次 RMSNorm**。

```python
self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)  # ~109 (@67f114a)
self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)  # ~110 (@67f114a)
xq, xk = self.q_norm(xq), self.k_norm(xk)                     # ~122 (@67f114a)
```

为什么?因为 Q 和 K 做点积 `Q @ K^T` 时,如果 Q/K 的某些维度数值特别大,scores 会爆炸,softmax 变 one-hot。QK-Norm 把 Q/K 的每个 head_dim 归一化,**让点积更稳定**。

> 注意:`q_norm` 和 `k_norm` 作用于 `head_dim`(96),不是 `hidden_size`(768)。因为归一化是在**每个头内部**进行的。V 不做 norm(V 不参与点积,只参与加权求和)。

In [ ]:
# QK-Norm 的效果演示
d_h = 96
q_norm = rmsnorm_compact  # 复用前面的 compact RMSNorm

# 不加 norm 的 Q
q = torch.randn(1, 8, 5, 96) * 10  # 故意放大 10 倍,Shape: (b,n_h,T,d_h)
k = torch.randn(1, 8, 5, 96) * 10

# Q @ K^T —— 不加 norm,scores 的数值范围很大
scores_no_norm = q @ k.transpose(-2, -1) / (d_h ** 0.5)
print(f"无 QK-Norm — scores 均值: {scores_no_norm.mean():.2f}, 标准差: {scores_no_norm.std():.2f}")
# scores 标准差可能 > 10,softmax 会很尖锐

# 加 norm 后
q_n = q_norm(q, torch.ones(d_h))
k_n = q_norm(k, torch.ones(d_h))
scores_norm = q_n @ k_n.transpose(-2, -1) / (d_h ** 0.5)
print(f"有 QK-Norm — scores 均值: {scores_norm.mean():.2f}, 标准差: {scores_norm.std():.2f}")
# scores 标准差大幅下降,softmax 更平滑

# 看 softmax 的分布
attn_no = F.softmax(scores_no_norm[0, 0], dim=-1)
attn_yes = F.softmax(scores_norm[0, 0], dim=-1)
print(f"无 norm — 最大注意力权重: {attn_no.max():.4f}")
print(f"有 norm — 最大注意力权重: {attn_yes.max():.4f}")  # 更接近均匀分布

### Part 3 小结:multi-head 三件套的 shape 节奏

到目前为止,我们给注意力加了三个工程优化,它们各自的 shape 变化是本章最需要记住的:

| 优化 | 输入 shape | 关键操作 | 输出 shape |
|---|---|---|---|
| **multi-head** | `(b, T, d)` | `view(b,T,n_h,d_h)` + `transpose(1,2)` | `(b, n_h, T, d_h)` |
| **RoPE** | `(b, n_h, T, d_h)` | `q*cos + rotate_half(q)*sin` | `(b, n_h, T, d_h)` 不变 |
| **GQA** | `(b, T, n_kv, d_h)` | `expand+reshape` → `repeat_kv` | `(b, T, n_kv*n_rep, d_h)` |
| **QK-Norm** | `(b, n_h, T, d_h)` | RMSNorm on `d_h` | 不变,只稳数值 |

> 记住这个节奏:**先 split heads(view+transpose)→ 再 RoPE → 再 repeat_kv → 最后算 attention**。minimind 的 forward 就是严格按这个顺序执行的(~118-136 (@67f114a))。

&nbsp;

---

## Part 4:KV Cache + 最终的 compact Attention

### 4.10 KV Cache:生成时为什么要缓存

自回归生成时,每生成一个新 token 就要重算整个序列的注意力。但**前面 token 的 K/V 并没有变** —— 重复计算纯属浪费。

**KV Cache** 把已经算过的 K/V 存起来,每步只算新 token 的 K/V,然后拼上去。

| 不用 cache | 用 cache |
|---|---|
| 第 $t$ 步重算 $1..t$ 所有 token 的 K/V | 第 $t$ 步只算第 $t$ 个 token 的 K/V |
| 总计算量 $O(T^2)$ | 总计算量 $O(T)$ |

In [ ]:
# KV Cache 的逻辑(model_minimind.py:~125-128 (@67f114a))

T_total = 10  # 假设要生成 10 个 token
n_kv = 4
d_h = 96

# 模拟逐 token 生成
past_k, past_v = None, None  # 初始 cache 为空

for step in range(T_total):
    x_new = torch.randn(1, 1, 768)  # Shape: (b, 1, d) — 只有一个新 token

    # 投影出新 token 的 K/V
    k_new = torch.randn(1, 1, n_kv, d_h)  # Shape: (b, 1, n_kv, d_h)
    v_new = torch.randn(1, 1, n_kv, d_h)  # Shape: (b, 1, n_kv, d_h)

    if past_k is not None:
        # 拼接历史 K/V:沿着 seq_len 维(dim=1)
        k_new = torch.cat([past_k, k_new], dim=1)  # Shape: (b, past_T+1, n_kv, d_h)
        v_new = torch.cat([past_v, v_new], dim=1)  # Shape: (b, past_T+1, n_kv, d_h)

    # 更新 cache
    past_k, past_v = k_new, v_new  # 为下一步缓存

    # 现在 k_new/v_new 包含了 0..step 的所有 K/V

print(f"生成 {T_total} 步后,KV cache shape: {past_k.shape}")
# (1, 10, 4, 96) — 累积了所有 10 个 token 的 K/V

# 对比:不用 cache 需要每步重算 1..step 的投影
total_ops_cache = sum(1 for _ in range(T_total))               # 10 次投影
total_ops_nocache = sum(t + 1 for t in range(T_total))         # 1+2+...+10 = 55 次投影
print(f"用 cache: {total_ops_cache} 次投影 | 不用 cache: {total_ops_nocache} 次投影")

> KV cache 把生成复杂度从 $O(T^2)$ 降到 $O(T)$,代价是**显存**。KV cache 显存 = `2 × n_kv × T × d_h × num_layers × bytes`。对 minimind: $2 × 4 × T × 96 × 8 × 2$(fp16) $= 12288T$ bytes。$T=4096$ 时约 50MB,$T=32768$ 时约 400MB。这也是为什么长上下文推理很费显存 —— KV cache 线性增长。

### 4.11 最终的 compact Attention:minimind 的 44 行实现

现在把前面所有零件组装起来。这就是 `model_minimind.py:~96-139 (@67f114a)` 的 `Attention` 类 —— minimind 真正跑的注意力:

```python
class Attention(nn.Module):
    def __init__(self, config):
        # ... 定义 Wq/Wk/Wv/Wo + q_norm/k_norm + flash attention 标志

    def forward(self, x, position_embeddings, past_key_value=None, ...):
        # 1. 投影 Q/K/V
        # 2. QK-Norm
        # 3. RoPE
        # 4. KV cache 拼接
        # 5. repeat_kv (GQA)
        # 6. attention 计算 (flash 或手写)
        # 7. 合并头 + 输出投影
```

下面用带详细 shape 注释的伪代码还原整个 forward:

In [ ]:
# === minimind Attention.forward 的 shape 全流程注释版 ===
# 参数:b=2( batch), T=5(seq_len), d=768, n_q=8, n_kv=4, d_h=96, n_rep=2

b, T, d = 2, 5, 768
n_q, n_kv, d_h, n_rep = 8, 4, 96, 2

x = torch.randn(b, T, d)  # 输入: (b, T, d_model)

# === Step 1: 投影 Q/K/V (~118-121 (@67f114a)) ===
Wq = nn.Linear(d, n_q * d_h, bias=False)    # (768, 768)
Wk = nn.Linear(d, n_kv * d_h, bias=False)   # (768, 384) — K/V 更小!
Wv = nn.Linear(d, n_kv * d_h, bias=False)   # (768, 384)
Wo = nn.Linear(n_q * d_h, d, bias=False)    # (768, 768)

xq = Wq(x).view(b, T, n_q, d_h)    # Shape: (b,T,d)->(b,T,n_q*d_h)->(b,T,n_q,d_h)=(2,5,8,96)
xk = Wk(x).view(b, T, n_kv, d_h)   # Shape: (b,T,d)->(b,T,n_kv*d_h)->(b,T,n_kv,d_h)=(2,5,4,96)
xv = Wv(x).view(b, T, n_kv, d_h)   # Shape: (b,T,d)->(b,T,n_kv*d_h)->(b,T,n_kv,d_h)=(2,5,4,96)

# === Step 2: QK-Norm (~122 (@67f114a)) ===
weight_norm = torch.ones(d_h)
xq = rmsnorm_compact(xq, weight_norm)  # Shape: (b,T,n_q,d_h) -> (b,T,n_q,d_h) — norm on d_h
xk = rmsnorm_compact(xk, weight_norm)  # Shape: (b,T,n_kv,d_h) -> (b,T,n_kv,d_h)

# === Step 3: RoPE (~123-124 (@67f114a)) ===
cos_pos = cos[:T]  # Shape: (T, d_h) = (5, 96)
sin_pos = sin[:T]  # Shape: (T, d_h) = (5, 96)
xq = apply_rope(xq, cos_pos, sin_pos)  # 需先 transpose,这里简化
xk = apply_rope(xk, cos_pos, sin_pos)  # Shape 不变: (b,T,n_*,d_h)

# === Step 4: transpose to (b, n_h, T, d_h) (~129 (@67f114a)) ===
xq = xq.transpose(1, 2)   # Shape: (b,T,n_q,d_h) -> (b,n_q,T,d_h) = (2,8,5,96)

# === Step 5: repeat_kv for GQA (~129 (@67f114a)) ===
xk = repeat_kv(xk, n_rep).transpose(1, 2)  # (b,T,n_kv,d_h)->(b,T,n_kv*n_rep,d_h)->(b,n_q,T,d_h)
xv = repeat_kv(xv, n_rep).transpose(1, 2)  # 同上,最终 (b, n_q, T, d_h) = (2,8,5,96)

# === Step 6: attention (~133-136 (@67f114a)) ===
scores = xq @ xk.transpose(-2, -1) / (d_h ** 0.5)  # Shape: (b,n_q,T,d_h)@(b,n_q,d_h,T)->(b,n_q,T,T)
# causal mask + softmax ...
attn = F.softmax(scores, dim=-1)                     # Shape: (b,n_q,T,T) -> (b,n_q,T,T)
out = attn @ xv                                      # Shape: (b,n_q,T,T)@(b,n_q,T,d_h)->(b,n_q,T,d_h)

# === Step 7: merge heads + output projection (~137-138 (@67f114a)) ===
out = out.transpose(1, 2)    # Shape: (b,n_q,T,d_h) -> (b,T,n_q,d_h) = (2,5,8,96)
out = out.reshape(b, T, -1)  # Shape: (b,T,n_q,d_h) -> (b,T,n_q*d_h) = (2,5,768)
out = Wo(out)                # Shape: (b,T,d) -> (b,T,d) = (2,5,768)

print("最终输出 shape:", out.shape)  # (2, 5, 768) — 和输入一样!
print("✅ 注意力层不改变 shape,只改变内容")

## Summary and takeaways

本章我们从零搭建了 minimind 的注意力机制,经历了 **naive → compact** 的完整演进:

| 概念 | naive 版本 | compact 版本 | 省了什么 |
|---|---|---|---|
| RMSNorm | 4 行展开 | `x * rsqrt(mean(x²)+eps)` | 代码简洁 |
| Self-attention | Q=K=V=x | Wq/Wk/Wv 投影 | 可学习 |
| Causal mask | masked_fill | `triu(1)` 一行 | 代码简洁 |
| Multi-head | 循环 8 次 | view+transpose 批量 | 速度 |
| RoPE | 逐对维度旋转 | cos/sin 预计算 | 速度 |
| GQA | 8 个 KV 头 | 4 个 + repeat_kv | 显存 |
| QK-Norm | 不加 | RMSNorm on d_h | 稳定性 |
| KV cache | 每步重算 | 拼接历史 | $O(T^2)→O(T)$ |

> **核心认知**:注意力不是一个公式 $\text{softmax}(QK^T/\sqrt{d})V$,而是**一层套一层的工程优化**。每一个优化(RoPE/GQA/QK-Norm/KV cache)都解决了朴素版本的一个具体问题。读 minimind 源码时,把它们当成「在朴素公式上打的补丁」来理解,就不会觉得复杂。

下一章我们看注意力之后的另一半:**FFN(SwiGLU)和 Transformer Block**(残差连接)。

- 精简复习版见 [`./attention.ipynb`](./attention.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)